In [2]:
!pip install requests
from google.colab import drive
drive.mount('/content/drive')
import requests
import json

Mounted at /content/drive


In [30]:
!pip install python-dotenv

# Fetch Stack Overflow Data

In [31]:
so_data = []

import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv('/content/drive/MyDrive/SpringForge/.env')
API_KEY = os.getenv('STACKOVERFLOW_API_KEY', '')

def fetch_so_runtime_with_fixes(pages=10, per_page=10, max_retries=3):
    runtime_keywords = ["exception", "error", "crash", "runtime", "nullpointer", "bean", "injection"]
    so_data = []

    session = requests.Session()
    retry_strategy = Retry(
        total=max_retries,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    for page in range(1, pages + 1):
        url = f"https://api.stackexchange.com/2.3/questions?page={page}&pagesize={per_page}&order=desc&sort=activity&tagged=spring-boot&filter=withbody&site=stackoverflow&hasaccepted=true"  # Added hasaccepted=true
        if API_KEY:
            url += f"&key={API_KEY}"

        for attempt in range(max_retries):
            try:
                response = session.get(url)
                if response.status_code == 200:
                    items = response.json().get('items', [])
                    for item in items:
                        q_id = item['question_id']
                        title = item['title'].lower()
                        body = item['body'].lower()[:500]

                        # Filter for runtime-related
                        if any(keyword in title or keyword in body for keyword in runtime_keywords):
                            time.sleep(1)
                            ans_url = f"https://api.stackexchange.com/2.3/questions/{q_id}/answers?order=desc&sort=activity&site=stackoverflow&filter=withbody"
                            if API_KEY:
                                ans_url += f"&key={API_KEY}"
                            ans_response = session.get(ans_url)
                            if ans_response.status_code == 200:
                                ans_items = ans_response.json().get('items', [])
                                accepted_ans = next((a['body'][:300] for a in ans_items if a.get('is_accepted')), None)
                                if accepted_ans:
                                    so_data.append({
                                        "id": len(so_data) + 1,
                                        "source": "StackOverflow",
                                        "title": item['title'],
                                        "content": body,
                                        "fix": accepted_ans
                                    })
                                    print(f"Fetched {len(so_data)}: {item['title'][:50]}...")
                    break
                elif response.status_code == 429:
                    retry_after = int(response.headers.get('Retry-After', 60))
                    print(f"Rate limited on page {page}, attempt {attempt+1}. Waiting {retry_after}s...")
                    time.sleep(retry_after)
                else:
                    print(f"Error on page {page}, attempt {attempt+1}: {response.status_code}")
                    time.sleep(2 ** attempt)
            except Exception as e:
                print(f"Exception on page {page}: {e}")
                time.sleep(2 ** attempt)

        time.sleep(2)

    return so_data

so_data = fetch_so_runtime_with_fixes()
print(f"Total runtime-related samples with fixes: {len(so_data)}")

Fetched 1: Missing constructor when using SELECT NEW DTO with...
Fetched 2: 502 bad gateway Elastic Beanstalk Spring Boot...
Fetched 3: Spring Boot/@JDBCTest - No qualifying bean of type...
Fetched 4: REVINFO table is missing the sequence &quot;revinf...
Fetched 5: Spring Security: AccessDeniedException with redire...
Fetched 6: PortUnreachableExceptions spamming log after updat...
Total runtime-related samples with fixes: 6


In [25]:
from bs4 import BeautifulSoup
import html

def clean_text(text):
    text = html.unescape(text)
    soup = BeautifulSoup(text, 'html.parser')
    text = soup.get_text()
    text = ' '.join(text.split())
    return text.strip()

cleaned_so_data = [
    {
        "id": entry["id"],
        "source": entry["source"],
        "title": clean_text(entry["title"]),
        "content": clean_text(entry["content"]),
        "fix": clean_text(entry["fix"])
    }
    for entry in so_data
]

for entry in cleaned_so_data[:5]:
    print(f"ID: {entry['id']}, Title: {entry['title'][:50]}..., Content: {entry['content'][:50]}..., Fix: {entry['fix'][:50]}...")

ID: 1, Title: Missing constructor when using SELECT NEW DTO with..., Content: i am using spring data jpa with hibernate 6. i wan..., Fix: In order for the constructor DTO syntax to work, H...
ID: 2, Title: 502 bad gateway Elastic Beanstalk Spring Boot..., Content: i deployed a spring boot app on aws elastic beanst..., Fix: It's because server is listening to 5000, Adding "...
ID: 3, Title: Spring Boot/@JDBCTest - No qualifying bean of type..., Content: bit of a spring boot rookie here, so appreciate an..., Fix: Hi :) What spring boot version do you have? Overal...
ID: 4, Title: REVINFO table is missing the sequence "revinfo_seq..., Content: i am migrating to springboot 3.0.1 and updated "hi..., Fix: You can solve it with this property: spring.jpa.pr...
ID: 5, Title: Spring Security: AccessDeniedException with redire..., Content: i follow the book pro spring security 6 and try to..., Fix: Use oauth2Login, not formLogin...


In [28]:
with open('/content/drive/MyDrive/SpringForge/so_runtime_samples.json', 'w') as f:
    json.dump(cleaned_so_data, f, indent=2)
print("Saved runtime samples to Drive!")

Saved runtime samples to Drive!
